In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import re
import time
from html import unescape
from collections import Counter

In [32]:
CATEGORY_MAP = {
    "economics": [
        "econ.EM",   # Econometrics
        "econ.GN",   # General Economics
        "econ.TH"    # Theoretical Economics
    ],
    "electrical_engineering": [
        "eess.AS",   # Audio and Speech Processing
        "eess.IV",   # Image and Video Processing
        "eess.SP",   # Signal Processing
        "eess.SY"    # Systems and Control
    ],
    "biology": [
        "q-bio.BM",  # Biomolecules
        "q-bio.GN",  # Genomics
        "q-bio.NC"   # Neurons and Cognition
    ]
}

In [33]:
def flatten_category_map(category_map):
    rows = []
    for broad_subject, subcats in category_map.items():
        for subcat in subcats:
            rows.append((broad_subject, subcat))
    return rows

flat_categories = flatten_category_map(CATEGORY_MAP)
flat_categories

[('economics', 'econ.EM'),
 ('economics', 'econ.GN'),
 ('economics', 'econ.TH'),
 ('electrical_engineering', 'eess.AS'),
 ('electrical_engineering', 'eess.IV'),
 ('electrical_engineering', 'eess.SP'),
 ('electrical_engineering', 'eess.SY'),
 ('biology', 'q-bio.BM'),
 ('biology', 'q-bio.GN'),
 ('biology', 'q-bio.NC')]

In [34]:
def fetch_primary_category_papers(category, target_primary_count=2000, batch_size=500, sort_by="submittedDate"):
    all_rows = []
    start = 0

    BASE_URL = "http://export.arxiv.org/api/query"

    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    stop_reason = "unknown"

    while len(all_rows) < target_primary_count and start < 30000:
        current_batch = min(batch_size, 2000)

        params = {
            "search_query": f"cat:{category}",
            "start": start,
            "max_results": current_batch,
            "sortBy": sort_by,
            "sortOrder": "descending"
        }

        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()

        root = ET.fromstring(response.text)
        entries = root.findall("atom:entry", ns)

        if not entries:
            stop_reason = "no more entries returned"
            break

        kept_this_batch = 0

        for entry in entries:
            title = entry.findtext("atom:title", default="", namespaces=ns)
            summary = entry.findtext("atom:summary", default="", namespaces=ns)
            published = entry.findtext("atom:published", default="", namespaces=ns)
            updated = entry.findtext("atom:updated", default="", namespaces=ns)
            paper_id = entry.findtext("atom:id", default="", namespaces=ns)

            authors = entry.findall("atom:author", ns)
            author_names = [a.findtext("atom:name", default="", namespaces=ns) for a in authors]

            categories_xml = entry.findall("atom:category", ns)
            category_terms = [c.attrib.get("term", "") for c in categories_xml]

            primary_category = ""
            primary_cat_elem = entry.find("arxiv:primary_category", ns)
            if primary_cat_elem is not None:
                primary_category = primary_cat_elem.attrib.get("term", "")

            if primary_category != category:
                continue

            row = {
                "id": paper_id,
                "title": title.strip(),
                "abstract": summary.strip(),
                "published": published,
                "updated": updated,
                "authors": ", ".join(author_names),
                "primary_category": primary_category,
                "all_categories": ", ".join(category_terms),
                "queried_category": category
            }

            all_rows.append(row)
            kept_this_batch += 1

            if len(all_rows) >= target_primary_count:
                stop_reason = "target reached"
                break

        print(
            f"{category}: start={start}, batch_returned={len(entries)}, "
            f"kept_primary={kept_this_batch}, total_kept={len(all_rows)}"
        )

        start += current_batch
        time.sleep(3)

    if start >= 30000 and len(all_rows) < target_primary_count:
        stop_reason = "hit start limit"

    print(f"{category}: STOPPED because -> {stop_reason}")
    return pd.DataFrame(all_rows)

In [62]:
TARGET_PER_PRIMARY_CATEGORY = 2000

all_dfs = []

for broad_subject, subcat in flat_categories:
    print(f"\nFetching primary papers for {subcat} under {broad_subject}...")
    
    temp_df = fetch_primary_category_papers(
        category=subcat,
        target_primary_count=TARGET_PER_PRIMARY_CATEGORY,
        batch_size=1000
    )
    
    temp_df["broad_subject"] = broad_subject
    all_dfs.append(temp_df)

df_raw = pd.concat(all_dfs, ignore_index=True)

print("\nTotal rows collected:", len(df_raw))


Fetching primary papers for econ.EM under economics...
econ.EM: start=0, batch_returned=1000, kept_primary=694, total_kept=694
econ.EM: start=1000, batch_returned=1000, kept_primary=727, total_kept=1421
econ.EM: start=2000, batch_returned=1000, kept_primary=579, total_kept=2000
econ.EM: STOPPED because -> target reached

Fetching primary papers for econ.GN under economics...
econ.GN: start=0, batch_returned=1000, kept_primary=750, total_kept=750
econ.GN: start=1000, batch_returned=1000, kept_primary=770, total_kept=1520
econ.GN: start=2000, batch_returned=1000, kept_primary=480, total_kept=2000
econ.GN: STOPPED because -> target reached

Fetching primary papers for econ.TH under economics...
econ.TH: start=0, batch_returned=1000, kept_primary=748, total_kept=748
econ.TH: start=1000, batch_returned=1000, kept_primary=718, total_kept=1466
econ.TH: start=2000, batch_returned=1000, kept_primary=534, total_kept=2000
econ.TH: STOPPED because -> target reached

Fetching primary papers for ee

In [63]:
print(df_raw["primary_category"].value_counts())

primary_category
econ.EM     2000
econ.GN     2000
econ.TH     2000
eess.AS     2000
eess.IV     2000
eess.SP     2000
eess.SY     2000
q-bio.BM    2000
q-bio.GN    2000
q-bio.NC    2000
Name: count, dtype: int64


In [64]:
df_raw.to_csv("data/arxiv_raw_metadata.csv", index=False)
df_raw.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject
0,http://arxiv.org/abs/2603.13823v1,Enhancing the Accuracy of Regional Input-Outpu...,Non-survey methods have been developed and app...,2026-03-14T08:12:29Z,2026-03-14T08:12:29Z,Shogo Fukui,econ.EM,econ.EM,econ.EM,economics
1,http://arxiv.org/abs/2603.13766v1,Estimating Earth's Temperature Response with T...,The long-term relationship between radiative f...,2026-03-14T05:33:16Z,2026-03-14T05:33:16Z,Justin Sun,econ.EM,econ.EM,econ.EM,economics
2,http://arxiv.org/abs/2603.13505v1,Testing the Exclusion Restriction in IV Models...,Instrumental variable (IV) methods rely critic...,2026-03-13T18:35:39Z,2026-03-13T18:35:39Z,Fernando Delbianco,econ.EM,econ.EM,econ.EM,economics
3,http://arxiv.org/abs/2603.12536v1,"Heterogeneous Elasticities, Aggregation, and R...",Economists often interpret estimates from line...,2026-03-13T00:34:54Z,2026-03-13T00:34:54Z,"Ellen Munroe, Alexander Newton, Meet Shah",econ.EM,econ.EM,econ.EM,economics
4,http://arxiv.org/abs/2603.12374v1,The Privacy-Utility Trade-Off of Location Trac...,Firms collect vast amounts of behavioral and g...,2026-03-12T18:52:18Z,2026-03-12T18:52:18Z,"Mohammad Mosaffa, Omid Rafieian",econ.EM,"econ.EM, cs.LG",econ.EM,economics


In [65]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = unescape(text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def simple_cleaning(df):
    df = df.copy()

    df["title_clean"] = df["title"].apply(clean_text)
    df["abstract_clean"] = df["abstract"].apply(clean_text)
    df["text"] = df["title_clean"] + " " + df["abstract_clean"]

    df["abstract_word_count"] = df["abstract_clean"].apply(lambda x: len(x.split()))
    df["title_word_count"] = df["title_clean"].apply(lambda x: len(x.split()))

    # remove duplicates by id
    df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    # remove very short abstracts
    df = df[df["abstract_word_count"] >= 20].reset_index(drop=True)

    return df

df = simple_cleaning(df_raw)
print("Rows b4 cleaning:", len(df_raw))
print("Rows after cleaning:", len(df))

Rows b4 cleaning: 20000
Rows after cleaning: 19988


In [66]:
# --- handle arXiv versions properly ---

df["base_id"] = df["id"].str.extract(r"(\d+\.\d+)")
df["version"] = df["id"].str.extract(r"v(\d+)$").astype(float)

print("Before dedup:", len(df))

df = df.sort_values(by=["base_id", "version"], ascending=[True, False])
df = df.drop_duplicates(subset="base_id", keep="first").reset_index(drop=True)

print("After dedup:", len(df))

Before dedup: 19988
After dedup: 19973


In [46]:
# compute base_id and version, count versions per paper
df['base_id'] = df['id'].str.extract(r'(\d+\.\d+)(?:v\d+)?$')[0]
df['version'] = df['id'].str.extract(r'v(\d+)$')[0].fillna('1').astype(int)

# count how many versions are present per base_id
version_counts = df.groupby('base_id').size().rename('n_versions')
print('Number of distinct base papers:', version_counts.shape[0])
print('How many base papers have multiple versions:', (version_counts>1).sum())
print('\nDistribution of version counts (n_versions: #base papers):')
print(version_counts.value_counts().sort_index())

# show a few examples of papers with multiple versions
multi = version_counts[version_counts>1].index
if len(multi):
    print('\nExample entries for papers with multiple versions:')
    display(df[df['base_id'].isin(multi)].sort_values(['base_id','version']).head(20))
else:
    print('\nNo multi-version papers found in current dataframe.')

# create a dataframe with latest version per base_id
df_latest = (
    df.sort_values(['base_id','version'], ascending=[True,False])
      .drop_duplicates(subset='base_id', keep='first')
      .reset_index(drop=True)
)
print('\nOriginal rows:', len(df))
print('Rows after keeping latest version only:', len(df_latest))

# assign back if you want to use latest-only going forward:
# df = df_latest.copy()


Number of distinct base papers: 19972
How many base papers have multiple versions: 0

Distribution of version counts (n_versions: #base papers):
n_versions
1    19972
Name: count, dtype: int64

No multi-version papers found in current dataframe.

Original rows: 19988
Rows after keeping latest version only: 19973


In [68]:
df.to_csv("data/arxiv_cleaned_metadata.csv", index=False)
df.head()

,id,title,abstract,published,updated,authors,primary_category,all_categories,queried_category,broad_subject,title_clean,abstract_clean,text,abstract_word_count,title_word_count,base_id,version
0,http://arxiv.org/abs/0704.1169v1,Holographic bound and protein linguistics,The holographic bound in physics constrains th...,2007-04-10T12:04:28Z,2007-04-10T12:04:28Z,"Dirson Jian Li, Shengli Zhang",q-bio.GN,"q-bio.GN, hep-th, q-bio.QM",q-bio.GN,biology,holographic bound and protein linguistics,the holographic bound in physics constrains th...,holographic bound and protein linguistics the ...,66,5,0704.1169,1.0
1,http://arxiv.org/abs/0704.2191v2,Mismatch Repair Error Implies Chargaff's Secon...,Chargaff's second parity rule holds empiricall...,2007-04-17T16:15:37Z,2007-09-20T15:46:34Z,Bo Deng,q-bio.GN,q-bio.GN,q-bio.GN,biology,mismatch repair error implies chargaff's secon...,chargaff's second parity rule holds empiricall...,mismatch repair error implies chargaff's secon...,155,8,0704.2191,2.0
2,http://arxiv.org/abs/0704.3826v2,Non-coding DNA programs express adaptation and...,Significant fraction (98.5% in humans) of most...,2007-04-29T05:32:50Z,2007-08-02T16:23:54Z,Mark Ya. Azbel,q-bio.GN,"q-bio.GN, cond-mat.other, nlin.AO, q-bio.OT, q...",q-bio.GN,biology,non-coding dna programs express adaptation and...,significant fraction (98.5% in humans) of most...,non-coding dna programs express adaptation and...,161,9,0704.3826,2.0
3,http://arxiv.org/abs/0705.1656v1,Chromatin Folding in Relation to Human Genome ...,Three-dimensional (3D) chromatin structure is ...,2007-05-11T14:12:05Z,2007-05-11T14:12:05Z,"Julio Mateos-Langerak, Osdilly Giromus, Wim de...",q-bio.GN,q-bio.GN,q-bio.GN,biology,chromatin folding in relation to human genome ...,three-dimensional (3d) chromatin structure is ...,chromatin folding in relation to human genome ...,226,8,0705.1656,1.0
4,http://arxiv.org/abs/0705.2143v1,Inverted and mirror repeats in model nucleotid...,We analytically and numerically study the prob...,2007-05-15T13:37:17Z,2007-05-15T13:37:17Z,"Fabrizio Lillo, Marco Spanó",q-bio.GN,"q-bio.GN, q-bio.QM",q-bio.GN,biology,inverted and mirror repeats in model nucleotid...,we analytically and numerically study the prob...,inverted and mirror repeats in model nucleotid...,79,8,0705.2143,1.0


In [69]:
print(df["abstract_word_count"].describe())

count    19973.000000
mean       165.876834
std         50.715755
min         20.000000
25%        130.000000
50%        162.000000
75%        200.000000
max        539.000000
Name: abstract_word_count, dtype: float64


In [70]:
df[["title", "primary_category", "published"]].sample(10, random_state=42)

,title,primary_category,published
6248,Change-Point Detection in Time Series Using Mi...,econ.EM,2024-08-11T01:02:58Z
15607,Dynamic Power Allocation For NOMA-Based Transm...,eess.SY,2025-11-28T16:27:50Z
18926,Is there a relationship between Mean Opinion S...,eess.IV,2026-02-19T02:17:13Z
15012,Improving the Generalisation of Learned Recons...,eess.IV,2025-11-16T18:57:13Z
16572,LIWhiz: A Non-Intrusive Lyric Intelligibility ...,eess.AS,2025-12-11T20:26:34Z
11791,Benchmarking spike source localization algorit...,q-bio.NC,2025-08-19T02:11:05Z
9072,DOSE : Drum One-Shot Extraction from Music Mix...,eess.AS,2025-04-25T08:12:35Z
9390,Diff-Unfolding: A Model-Based Score Learning F...,eess.IV,2025-05-16T15:57:19Z
10210,Structural Similarity-Inspired Unfolding for L...,eess.IV,2025-06-13T14:29:40Z
9226,Guidance for Intra-cardiac Echocardiography Ma...,eess.IV,2025-05-08T02:48:30Z
